# EasyRAG Document Quality And Edge Cases

## Chapter position

This chapter sits before indexing. Raw files or in-memory text become canonical `Document` objects here, so mistakes at this layer tend to echo through every later stage.

## Learning question

How does EasyRAG surface weak inputs before they turn into bad chunks or confusing retrieval results?

## Success criteria

- build a small batch with empty, duplicate, short, missing-path, and PDF-like edge cases
- inspect `annotate_document_quality()` before insertion
- compare those flags with insertion stats and document-status records

## Source code anchors

- `easyrag.rag.indexing.quality.annotate_document_quality`
- `easyrag.rag.indexing.prepare.prepare_documents_for_insert_with_report`
- `easyrag.rag.orchestrator.EasyRAG.ainsert_documents`
- `easyrag.rag.storage.doc_status.BaseDocStatusStorage.get_status`


## Method principles

- `easyrag.rag.indexing.quality.annotate_document_quality`: This pass adds quality flags to normalized documents before indexing. It catches empties, duplicates, very short content, visual-only PDFs, and missing paths early enough to keep them inspectable.
- `easyrag.rag.indexing.prepare.prepare_documents_for_insert_with_report`: This is the same preparation path plus a small report about skipped empty inputs. It is useful when you want to see what normalization removed before insertion begins.
- `easyrag.rag.orchestrator.EasyRAG.ainsert_documents`: This insertion path assumes you already built canonical `Document` objects. It preserves that metadata and sends the documents straight into the normal ingestion pipeline.
- `easyrag.rag.storage.doc_status.BaseDocStatusStorage.get_status`: This storage contract method reads the status record for one document. In practice it tells you whether a document was indexed and which quality flags or chunk counts were recorded.


## How the code fits together

The flow in this notebook is messy inputs -> quality annotations -> insertion report -> status storage. The goal is not to show every internal detail at once. The goal is to keep the boundary for this stage visible enough that later behavior still feels explainable. If a result looks odd, the intermediate objects in this notebook should tell you where to look next.

In [ ]:
# ruff: noqa: E402,F401,F811
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "easyrag").exists():
        REPO_ROOT = candidate.resolve()
        if str(REPO_ROOT) not in sys.path:
            sys.path.insert(0, str(REPO_ROOT))
        break
else:
    raise RuntimeError("Could not locate the EasyRAG repository root.")

import json
import os
import tempfile
import time
from pathlib import Path
from pprint import pprint
from unittest import mock

from easyrag.rag import AnswerParam, EasyRAG, EvalCase, QueryParam
from easyrag.support.async_utils import run_sync
from notebooks._utils import (
    managed_demo_rag,
    print_json as _print_json,
    production_backends_ready,
    provider_ready as can_use_openai_compatible_models,
    skip_message,
    stub_embedding as _stub_embedding,
    stub_query_model as _stub_query_model,
    stub_reranker as _stub_reranker,
)

from easyrag.rag.indexing.prepare import prepare_documents_for_insert_with_report
from easyrag.rag.indexing.quality import annotate_document_quality
from easyrag.support.optional_deps import Document

## What this cell is proving

We will build the batch in two layers. First, we use the normal preparation helper on raw texts. Then we add one manual PDF-like `Document` so we can trigger the `pdf_visual_only` rule explicitly.

In [ ]:
raw_texts = [
    "\x00 \n\n",
    "tiny doc",
    "tiny doc",
    "Service map with a missing path field.",
]
raw_ids = ["doc::empty", "doc::one", "doc::two", "doc::missing-path"]
raw_paths = ["docs/empty.md", "docs/one.md", "docs/two.md", ""]

prepared_documents, preparation_report = prepare_documents_for_insert_with_report(
    raw_texts, ids=raw_ids, file_paths=raw_paths
)
pdf_like_document = Document(
    page_content="scanned pdf page 1 figure 1 table 2",
    metadata={
        "doc_id": "doc::scan",
        "path": "docs/scan.pdf",
        "relative_path": "docs/scan.pdf",
        "title": "scan",
        "source_type": "pdf",
        "has_visual_content": True,
    },
)

quality_batch = list(prepared_documents) + [pdf_like_document]
annotated_documents, quality_report = annotate_document_quality(quality_batch)

print("Preparation report from raw inputs")
_print_json(preparation_report)
print("\nAnnotated document metadata")
for document in annotated_documents:
    _print_json(document.metadata)
    print(document.page_content)
    print()
print("Quality report")
_print_json(quality_report)

## Why this output looks like this

A few edge cases are worth calling out. The blank input disappears before insertion. Duplicate text remains indexable, but it is flagged. Very short documents also remain indexable, which is useful for diagnostics. The PDF-like document survives too, but it carries an explicit warning so you can choose whether to trust it.

In [ ]:
quality_tmp = tempfile.TemporaryDirectory()
quality_rag = EasyRAG(
    working_dir=quality_tmp.name,
    workspace="quality-demo",
    embedding_func=_stub_embedding,
    query_model_func=_stub_query_model,
)
run_sync(quality_rag.initialize_storages())

insert_stats = run_sync(quality_rag.ainsert_documents(quality_batch))
status_one = run_sync(quality_rag.doc_status_storage.get_status("doc::one"))
status_scan = run_sync(quality_rag.doc_status_storage.get_status("doc::scan"))

print("Insertion stats")
_print_json(insert_stats)
print("\nStatus for doc::one")
_print_json(status_one)
print("\nStatus for doc::scan")
_print_json(status_scan)

## What this cell is proving

The provider-backed path should preserve the same contract even when the backend behavior is less deterministic.

The provider-backed path uses the same edge-case batch. The point is not to get a better answer. The point is to show that document-quality accounting still happens before any external model call.

In [ ]:
if not can_use_openai_compatible_models():
    print("Skipping provider-backed path because OPENAI-compatible config is not set.")
else:
    provider_tmp = tempfile.TemporaryDirectory()
    provider_rag = EasyRAG(
        working_dir=provider_tmp.name, workspace="provider-quality-demo"
    )
    try:
        run_sync(provider_rag.initialize_storages())
        provider_stats = run_sync(provider_rag.ainsert_documents(quality_batch))
        print("Provider-backed insertion stats")
        _print_json(provider_stats)
    except Exception as exc:
        print(f"Provider-backed path was skipped due to runtime error: {exc}")
    finally:
        try:
            run_sync(provider_rag.finalize_storages())
        finally:
            provider_tmp.cleanup()

## Common mistakes / debugging cues

- If a later stage looks wrong, inspect `doc_id`, logical path, title, and normalized text here first.
- Do not confuse canonical `Document` preparation with actual index writes. They are different boundaries.
- Noisy or empty documents usually create problems long before retrieval. Loading and quality checks are where you catch them cheaply.

## Takeaway

The important distinction is between *skip* and *flag*. Empty-after-normalization content is skipped because there is nothing meaningful to index. Very short, duplicate, missing-path, and PDF-visual-only documents are still indexable, but the notebook keeps those warnings visible so you can decide whether the corpus is healthy enough for retrieval experiments.

In [ ]:
run_sync(quality_rag.finalize_storages())
quality_tmp.cleanup()
print("Cleaned up the deterministic quality workspace.")

## Where to go next

- Continue with [03_01_chunking_principles.ipynb](../03_indexing/03_01_chunking_principles.ipynb) to see how cleaned documents become chunk candidates.
- Read [02-data-loading-overview.md](../../docs/02-data-loading-overview.md) for the loading-stage explanation behind these quality checks.
- Revisit [02_04_normalization_and_cleaning.ipynb](02_04_normalization_and_cleaning.ipynb) if you want to separate normalization effects from quality-flag logic.